In [ ]:
!pip install pennylane
!pip install torch
!pip install torchinfo

import matplotlib
from matplotlib import pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary

from minitorch import NumpyDataset, train, set_seed

import pennylane as qml

rs = 1234  # Semilla aleatoria

set_seed(rs)

Creamos un conjunto de datos sencillo y los dividimos en entrenamiento y test.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=200, centers = [[0,2],[2,0]], random_state = rs)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=rs)
X_train, X_vali, y_train, y_vali = train_test_split(X_train, y_train, test_size=0.2, stratify = y_train, random_state=rs)
plt.scatter(X[:, 0], X[:, 1], c = y, cmap=matplotlib.colors.ListedColormap(["red","green"]));

Como vamos a usar *angle embedding*, escalamos los datos al intervalo $[0,\pi]$.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0,np.pi))
X_train = scaler.fit_transform(X_train)
X_vali = scaler.transform(X_vali)
X_test = scaler.transform(X_test)

In [ ]:
tr_data = NumpyDataset(X_train, y_train)
val_data = NumpyDataset(X_vali, y_vali)
test_data = NumpyDataset(X_test, y_test)

In [ ]:
n_qubits = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def qnode(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

n_layers = 1
weight_shapes = {"weights": (n_layers, n_qubits, 3)}

qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)

model = nn.Sequential(
    nn.Linear(2, n_qubits),
    nn.ReLU(),
    qlayer,
    nn.Linear(n_qubits, 1),
    nn.Sigmoid()
)

opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss =  F.binary_cross_entropy

In [ ]:
history = train(model=model, loss=loss, opt=opt, tr_data=tr_data, val_data=val_data, max_epochs=100, patience=10)

In [ ]:
history.plot_losses()
history.plot_accuracies()

In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(model(test_data.x).detach().numpy() >= 0.5, test_data.y)

In [ ]:
summary(model)

Ahora, usamos dos repeticiones de la forma variacional.

In [ ]:
n_qubits = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def qnode(inputs, weights):
    qml.templates.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

n_layers = 2
weight_shapes = {"weights": (n_layers, n_qubits, 3)}

qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)

model = nn.Sequential(
    nn.Linear(2, n_qubits),
    nn.ReLU(),
    qlayer,
    nn.Linear(n_qubits, 1),
    nn.Sigmoid()
)

opt = torch.optim.Adam(model.parameters(), lr=0.05)
loss =  F.binary_cross_entropy

In [ ]:
history = train(model=model, loss=loss, opt=opt, tr_data=tr_data, val_data=val_data, max_epochs=100, patience=10)

In [ ]:
history.plot_losses()
history.plot_accuracies()

In [ ]:
accuracy_score(model(test_data.x).detach().numpy() >= 0.5, test_data.y)

In [ ]:
summary(model)